In [3]:

# PROJECT 13: PROFITABILITY DECLINE INVESTIGATION AGENT
# Synthetic cross-functional business analysis + interactive dashboard

!pip -q install gradio pandas numpy plotly scikit-learn

import numpy as np
import pandas as pd
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# ============================================================
# 1. GENERATE SYNTHETIC QUARTERLY COMPANY DATA
# ============================================================

rng = np.random.default_rng(130)
quarters = pd.period_range("2023Q1", periods=12, freq="Q")
rows = []

for i, quarter in enumerate(quarters):
    # Baseline business indicators
    marketing_spend = 180 + 4*i + rng.normal(0, 8)
    conversion_rate = 4.2 - 0.025*i + rng.normal(0, 0.12)
    customer_acquisition_cost = 42 + 0.8*i + rng.normal(0, 2)

    headcount = 300 + int(2*i + rng.normal(0, 3))
    overtime_hours = max(100, 480 + 7*i + rng.normal(0, 35))
    absenteeism = max(2, 3.5 + 0.08*i + rng.normal(0, 0.25))
    productivity_index = 100 - 0.35*i + rng.normal(0, 1.5)

    production_cost = 490 + 8*i + rng.normal(0, 15)
    logistics_cost = 125 + 2.8*i + rng.normal(0, 7)
    waste_rate = max(1, 3.0 + 0.09*i + rng.normal(0, 0.25))
    capacity_utilization = 82 - 0.15*i + rng.normal(0, 2)

    # Revenue increases initially, then weakens as marketing efficiency
    # and conversion decline. Latest three quarters have additional shocks.
    revenue = 1500 + 24*i + 2.1*marketing_spend + rng.normal(0, 25)
    revenue += (conversion_rate - 4.2) * 100

    if i >= 9:
        conversion_rate -= 0.18
        customer_acquisition_cost += 5
        overtime_hours += 70
        absenteeism += 0.8
        production_cost += 38
        logistics_cost += 18
        waste_rate += 0.7
        revenue -= 55

    marketing_spend = max(0, marketing_spend)
    revenue = max(500, revenue)

    # Costs grow faster than revenue during the last three quarters.
    total_cost = (
        790 + 15*i
        + marketing_spend * 0.45
        + production_cost * 0.35
        + logistics_cost
        + headcount * 0.35
        + overtime_hours * 0.035
        + absenteeism * 12
        + waste_rate * 12
        + rng.normal(0, 12)
    )

    if i >= 9:
        total_cost += 90

    operating_profit = revenue - total_cost
    profit_margin = operating_profit / revenue * 100

    rows.append({
        "Quarter": str(quarter),
        "Marketing_Spend_k": round(marketing_spend, 2),
        "Conversion_Rate_pct": round(conversion_rate, 2),
        "Customer_Acquisition_Cost": round(customer_acquisition_cost, 2),
        "Headcount": headcount,
        "Overtime_Hours": round(overtime_hours, 1),
        "Absenteeism_pct": round(absenteeism, 2),
        "Productivity_Index": round(productivity_index, 2),
        "Production_Cost_k": round(production_cost, 2),
        "Logistics_Cost_k": round(logistics_cost, 2),
        "Waste_Rate_pct": round(waste_rate, 2),
        "Capacity_Utilization_pct": round(capacity_utilization, 2),
        "Revenue_k": round(revenue, 2),
        "Total_Operating_Cost_k": round(total_cost, 2),
        "Operating_Profit_k": round(operating_profit, 2),
        "Profit_Margin_pct": round(profit_margin, 2)
    })

synthetic_data = pd.DataFrame(rows)

# Departmental mapping for the dashboard
DEPARTMENTS = {
    "Marketing": [
        "Marketing_Spend_k", "Conversion_Rate_pct",
        "Customer_Acquisition_Cost"
    ],
    "HR": [
        "Headcount", "Overtime_Hours", "Absenteeism_pct",
        "Productivity_Index"
    ],
    "Operations": [
        "Production_Cost_k", "Logistics_Cost_k",
        "Waste_Rate_pct", "Capacity_Utilization_pct"
    ],
    "Finance": [
        "Revenue_k", "Total_Operating_Cost_k",
        "Operating_Profit_k", "Profit_Margin_pct"
    ]
}

# ============================================================
# 2. BUSINESS ANALYSIS ENGINE
# ============================================================

TARGETS = ["Operating_Profit_k", "Profit_Margin_pct"]

def investigate(data, target="Operating_Profit_k"):
    data = data.copy()

    required = ["Quarter", target]
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))

    data[target] = pd.to_numeric(data[target], errors="coerce")
    data = data.dropna(subset=[target]).reset_index(drop=True)

    if len(data) < 6:
        raise ValueError("At least six valid quarterly records are required.")

    latest = data.iloc[-1]
    previous = data.iloc[-2]

    # Identify the latest consecutive decline streak
    decline_streak = 0
    for i in range(len(data)-1, 0, -1):
        if data.loc[i, target] < data.loc[i-1, target]:
            decline_streak += 1
        else:
            break

    # Compare latest three quarters with the preceding three.
    if len(data) >= 6:
        recent = data.tail(3)
        prior = data.iloc[-6:-3]
        recent_avg = recent[target].mean()
        prior_avg = prior[target].mean()
        period_change = (
            (recent_avg-prior_avg)/abs(prior_avg)*100
            if prior_avg != 0 else np.nan
        )
    else:
        recent = data.tail(3)
        prior = data.iloc[:-3]
        recent_avg = recent[target].mean()
        prior_avg = prior[target].mean()
        period_change = np.nan

    # Rank KPI associations and model-based feature importance.
    excluded = set(TARGETS + ["Revenue_k", "Total_Operating_Cost_k"])
    features = [
        c for c in data.select_dtypes(include=np.number).columns
        if c not in excluded and data[c].nunique() > 1
    ]

    ranked_rows = []
    for col in features:
        valid = data[[col, target]].dropna()
        if len(valid) >= 6 and valid[col].nunique() > 1:
            corr = valid[col].corr(valid[target])
            if pd.notna(corr):
                ranked_rows.append({
                    "KPI": col,
                    "Department": next(
                        (d for d, cols in DEPARTMENTS.items() if col in cols),
                        "Other"
                    ),
                    "Correlation": corr,
                    "Absolute_Correlation": abs(corr)
                })

    ranking = pd.DataFrame(ranked_rows)
    if not ranking.empty:
        ranking = ranking.sort_values(
            "Absolute_Correlation", ascending=False
        )

    # A small model is used only as an exploratory ranking aid.
    # With quarterly data, feature importance is highly uncertain.
    model_importance = pd.DataFrame()
    if len(data) >= 8 and len(features) >= 2:
        X = data[features].replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.median()).fillna(0)
        y = data[target]
        model = RandomForestRegressor(
            n_estimators=150, max_depth=3, random_state=130
        )
        model.fit(X, y)
        model_importance = pd.DataFrame({
            "KPI": features,
            "Model_Importance": model.feature_importances_
        }).sort_values("Model_Importance", ascending=False)

    # Executive narrative and operational signals
    narrative = []
    narrative.append(
        f"**Latest quarter:** {latest['Quarter']}."
    )
    narrative.append(
        f"**{target.replace('_', ' ')}:** {latest[target]:,.2f}."
    )

    if decline_streak >= 3:
        narrative.append(
            f"**Profitability alert:** {decline_streak} consecutive "
            "quarter-on-quarter declines were detected."
        )
    elif decline_streak > 0:
        narrative.append(
            f"**Trend alert:** {decline_streak} consecutive "
            "quarter-on-quarter declines were detected."
        )
    else:
        narrative.append(
            "No current consecutive decline streak was detected "
            "for the selected financial outcome."
        )

    if pd.notna(period_change):
        narrative.append(
            f"The latest three-quarter average is {period_change:+.1f}% "
            "versus the previous three-quarter average."
        )

    if not ranking.empty:
        for _, row in ranking.head(4).iterrows():
            direction = (
                "positive" if row["Correlation"] > 0 else "negative"
            )
            narrative.append(
                f"**{row['Department']} — {row['KPI'].replace('_', ' ')}:** "
                f"{direction} association with the selected outcome "
                f"(r = {row['Correlation']:+.2f})."
            )

    # Recommendations based on observed recent changes.
    recommendations = []
    if len(data) >= 6:
        recent = data.tail(3)
        prior = data.iloc[-6:-3]

        checks = [
            ("Marketing_Spend_k",
             "Marketing", "Review campaign-level return, channel mix and acquisition costs."),
            ("Conversion_Rate_pct",
             "Marketing", "Investigate funnel conversion and lead quality before increasing campaign budgets."),
            ("Customer_Acquisition_Cost",
             "Marketing", "Compare acquisition cost by channel and customer segment."),
            ("Overtime_Hours",
             "HR", "Review overtime drivers, shift coverage and workload allocation."),
            ("Absenteeism_pct",
             "HR", "Investigate absence patterns and staffing coverage at team level."),
            ("Productivity_Index",
             "HR", "Validate productivity trends alongside workload, quality and staffing."),
            ("Production_Cost_k",
             "Operations", "Break down unit costs, material usage and production inefficiencies."),
            ("Logistics_Cost_k",
             "Operations", "Compare carrier rates, route efficiency and expedited-shipping costs."),
            ("Waste_Rate_pct",
             "Operations", "Inspect scrap, rework and quality-control losses."),
            ("Capacity_Utilization_pct",
             "Operations", "Check capacity planning and underutilized production resources.")
        ]

        for col, department, action in checks:
            if col in data.columns:
                recent_avg = recent[col].mean()
                prior_avg = prior[col].mean()
                if prior_avg != 0:
                    delta = (recent_avg-prior_avg)/abs(prior_avg)*100
                    harmful = (
                        (col in ["Marketing_Spend_k", "Customer_Acquisition_Cost",
                                 "Overtime_Hours", "Absenteeism_pct",
                                 "Production_Cost_k", "Logistics_Cost_k",
                                 "Waste_Rate_pct"] and delta > 5)
                        or
                        (col in ["Conversion_Rate_pct", "Productivity_Index",
                                 "Capacity_Utilization_pct"] and delta < -5)
                    )
                    if harmful:
                        recommendations.append(
                            f"**{department}:** {action} "
                            f"({delta:+.1f}% across the compared periods)."
                        )

    if not recommendations:
        recommendations.append(
            "Conduct a cross-functional review of the strongest KPI signals, "
            "validate them with departmental owners, and test one measurable "
            "improvement before scaling."
        )

    recommendations.append(
        "**Finance:** reconcile the reported profit decline against the "
        "general ledger, revenue mix, pricing, discounts and cost allocations."
    )
    recommendations.append(
        "**Governance:** correlations and model importance are investigative "
        "clues, not proof of causation. Confirm findings before taking action."
    )

    return data, ranking, model_importance, narrative, recommendations, decline_streak, period_change

# ============================================================
# 3. PREMIUM EXECUTIVE DASHBOARD
# ============================================================

CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Space+Grotesk:wght@500;600;700&display=swap');
.gradio-container {
    max-width: 1550px !important;
    background: #0c1015 !important;
    color: #edf2f7 !important;
    font-family: 'DM Sans', sans-serif !important;
}
h1,h2,h3 { font-family: 'Space Grotesk', sans-serif !important; }
.hero {
    background: linear-gradient(120deg,#1a2730,#171d27 60%,#30231d);
    border: 1px solid #35404b; border-radius: 18px;
    padding: 26px; margin-bottom: 18px;
}
.eyebrow { color:#65e5c1; font-size:11px; font-weight:700; letter-spacing:2px; }
.hero p,.muted { color:#aab6c4; line-height:1.7; }
.kpis { display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:12px; margin:14px 0; }
.kpi { background:#151b23; border:1px solid #2a3440; border-radius:14px; padding:19px; }
.kpi span { color:#9aa8b8; font-size:12px; display:block; }
.kpi strong { font:700 25px 'Space Grotesk',sans-serif; display:block; margin:10px 0; color:#f2f6fa; }
.kpi small { color:#65e5c1; }
.panel { background:#151b23; border:1px solid #2a3440; border-radius:14px; padding:19px; margin:10px 0; }
.panel li { line-height:1.75; margin:8px 0; color:#d7e0e9; }
button.primary { background:#65e5c1 !important; color:#071510 !important; font-weight:700 !important; border:0 !important; }
button { border-radius:10px !important; }
.note { color:#9aa8b8; font-size:12px; line-height:1.7; }
@media(max-width:800px) { .kpis {grid-template-columns:repeat(2,minmax(0,1fr));} }
"""

def render_analysis(upload, target, department):
    data = pd.read_csv(upload) if upload else synthetic_data.copy()
    analyzed, ranking, model_importance, narrative, recommendations, streak, period_change = investigate(data, target)

    latest = analyzed.iloc[-1]
    previous = analyzed.iloc[-2]

    if department != "All departments" and not ranking.empty:
        view_ranking = ranking[ranking["Department"] == department].copy()
        if view_ranking.empty:
            view_ranking = ranking.copy()
    else:
        view_ranking = ranking.copy()

    revenue = latest.get("Revenue_k", np.nan)
    profit = latest.get("Operating_Profit_k", np.nan)
    margin = latest.get("Profit_Margin_pct", np.nan)
    costs = latest.get("Total_Operating_Cost_k", np.nan)

    def fmt_money(v):
        return f"${v:,.1f}k" if pd.notna(v) else "N/A"

    cards = f"""
    <div class="kpis">
      <div class="kpi"><span>REVENUE</span><strong>{fmt_money(revenue)}</strong><small>Latest quarter</small></div>
      <div class="kpi"><span>OPERATING PROFIT</span><strong>{fmt_money(profit)}</strong><small>Latest quarter</small></div>
      <div class="kpi"><span>PROFIT MARGIN</span><strong>{f'{margin:.1f}%' if pd.notna(margin) else 'N/A'}</strong><small>Operating profit / revenue</small></div>
      <div class="kpi"><span>OPERATING COST</span><strong>{fmt_money(costs)}</strong><small>Latest quarter</small></div>
    </div>
    <p class="note">Illustrative synthetic data unless a CSV is uploaded. Currency values are expressed in thousands ($k).</p>
    """

    narrative_html = (
        '<div class="panel"><div class="eyebrow">INVESTIGATION SUMMARY</div><ul>'
        + "".join(f"<li>{x.replace('**','<b>',1).replace('**', '</b>',1) if x.count('**') >= 2 else x}</li>" for x in narrative)
        + '</ul></div>'
        '<div class="panel"><div class="eyebrow">RECOMMENDED INVESTIGATION PLAN</div><ul>'
        + "".join(f"<li>{x}</li>" for x in recommendations)
        + '</ul><p class="note">This is decision support, not an automated final diagnosis.</p></div>'
    )

    # Financial trend
    trend = go.Figure()
    trend.add_trace(go.Scatter(
        x=analyzed["Quarter"], y=analyzed[target],
        mode="lines+markers+text",
        line=dict(color="#65e5c1", width=3),
        marker=dict(size=8, color="#65e5c1"),
        fill="tozeroy", fillcolor="rgba(101,229,193,0.08)",
        name=target.replace("_"," ")
    ))
    trend.update_layout(
        title=f"{target.replace('_',' ')} · quarterly trend",
        template="plotly_dark", paper_bgcolor="#151b23",
        plot_bgcolor="#151b23", font_color="#edf2f7",
        margin=dict(l=25,r=20,t=55,b=30)
    )
    trend.update_xaxes(gridcolor="#2a3440")
    trend.update_yaxes(gridcolor="#2a3440")

    # Cross-functional driver chart
    if view_ranking.empty:
        drivers = go.Figure()
        drivers.add_annotation(text="No eligible KPI associations found", showarrow=False)
    else:
        chart_data = view_ranking.head(10).sort_values("Correlation")
        colors = [
            "#65e5c1" if v >= 0 else "#ff927e"
            for v in chart_data["Correlation"]
        ]
        drivers = go.Figure(go.Bar(
            x=chart_data["Correlation"],
            y=chart_data["KPI"].str.replace("_"," "),
            orientation="h", marker_color=colors,
            text=chart_data["Correlation"].map(lambda v: f"{v:+.2f}"),
            textposition="outside"
        ))
        drivers.add_vline(x=0, line_color="#718096")
    drivers.update_layout(
        title="Potential cross-functional KPI relationships",
        template="plotly_dark", paper_bgcolor="#151b23",
        plot_bgcolor="#151b23", font_color="#edf2f7",
        xaxis_title="Correlation with selected financial outcome",
        margin=dict(l=25,r=40,t=55,b=30)
    )
    drivers.update_xaxes(range=[-1.2, 1.2], gridcolor="#2a3440")
    drivers.update_yaxes(gridcolor="#2a3440")

    # Compare average KPI values in previous three quarters vs latest three.
    compare_cols = [
        c for c in DEPARTMENTS.get(department, [])
        if c in analyzed.columns
    ] if department != "All departments" else [
        c for cols in DEPARTMENTS.values() for c in cols if c in analyzed.columns
    ]

    compare_rows = []
    if len(analyzed) >= 6:
        old = analyzed.iloc[-6:-3]
        new = analyzed.tail(3)
        for col in compare_cols:
            a, b = old[col].mean(), new[col].mean()
            pct = (b-a)/abs(a)*100 if a != 0 else np.nan
            compare_rows.append({
                "Department": next((d for d, cols in DEPARTMENTS.items() if col in cols), "Other"),
                "KPI": col,
                "Previous 3Q Average": round(a, 2),
                "Latest 3Q Average": round(b, 2),
                "Change_pct": round(pct, 2) if pd.notna(pct) else np.nan
            })
    comparison = pd.DataFrame(compare_rows)

    if not comparison.empty:
        cross_chart = px.bar(
            comparison, x="KPI", y="Change_pct", color="Department",
            barmode="group", title="KPI change: latest 3 quarters vs previous 3",
            color_discrete_sequence=["#65e5c1","#83aaff","#ffb86b","#ff927e"]
        )
        cross_chart.update_layout(
            template="plotly_dark", paper_bgcolor="#151b23",
            plot_bgcolor="#151b23", font_color="#edf2f7",
            margin=dict(l=25,r=20,t=55,b=110)
        )
        cross_chart.update_xaxes(tickangle=-35, gridcolor="#2a3440")
        cross_chart.update_yaxes(gridcolor="#2a3440")
    else:
        cross_chart = go.Figure()
        cross_chart.add_annotation(text="At least six quarters are needed for period comparison.", showarrow=False)

    # Model feature importance, if available
    if not model_importance.empty:
        mi = model_importance.head(10).sort_values("Model_Importance")
        model_chart = px.bar(
            mi, x="Model_Importance", y="KPI", orientation="h",
            title="Exploratory model feature importance",
            color_discrete_sequence=["#83aaff"]
        )
        model_chart.update_layout(
            template="plotly_dark", paper_bgcolor="#151b23",
            plot_bgcolor="#151b23", font_color="#edf2f7"
        )
    else:
        model_chart = go.Figure()
        model_chart.add_annotation(
            text="More quarterly observations are needed for this exploratory model.",
            showarrow=False
        )

    return (
        cards, narrative_html, trend, drivers, cross_chart, model_chart,
        analyzed.tail(12), view_ranking, comparison
    )

with gr.Blocks(theme=gr.themes.Base(), css=CSS, title="Profitability Investigation Centre") as app:
    gr.HTML("""
    <div class="hero">
      <div class="eyebrow">CROSS-FUNCTIONAL BUSINESS INTELLIGENCE</div>
      <h1 style="font-size:34px;margin:10px 0">Profitability Decline Investigation</h1>
      <p>
        Find the connections behind declining profit.
        Investigate marketing efficiency, workforce costs, operational performance
        and financial outcomes in one decision-support workspace.
      </p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1, min_width=230):
            gr.Markdown("## Investigation controls")
            upload = gr.File(
                label="Upload quarterly company data (CSV)",
                file_types=[".csv"], type="filepath"
            )
            target = gr.Dropdown(
                choices=TARGETS, value="Operating_Profit_k",
                label="Financial outcome"
            )
            department = gr.Dropdown(
                choices=["All departments","Marketing","HR","Operations","Finance"],
                value="All departments", label="Department focus"
            )
            run = gr.Button("✦ Investigate profitability", variant="primary")
            gr.Markdown("""
            <div class="panel">
              <div class="eyebrow">ANALYSIS WORKFLOW</div>
              <p class="note">
                01 · Detect decline<br>
                02 · Compare quarterly periods<br>
                03 · Rank KPI relationships<br>
                04 · Investigate departments<br>
                05 · Recommend next actions
              </p>
            </div>
            <p class="note">
              Default data is synthetic. CSV requires Quarter and a selected
              financial outcome. At least six quarterly records are recommended.
            </p>
            """)

        with gr.Column(scale=4):
            cards_out = gr.HTML()
            with gr.Tabs():
                with gr.Tab("Executive Brief"):
                    brief_out = gr.HTML()
                with gr.Tab("Profit Trend"):
                    trend_out = gr.Plot()
                with gr.Tab("Potential Drivers"):
                    drivers_out = gr.Plot()
                    ranking_out = gr.Dataframe(label="KPI association ranking", interactive=False)
                with gr.Tab("Cross-Functional View"):
                    cross_out = gr.Plot()
                    comparison_out = gr.Dataframe(label="Quarterly KPI comparison", interactive=False)
                with gr.Tab("Model Signals"):
                    model_out = gr.Plot()
                with gr.Tab("Data Explorer"):
                    data_out = gr.Dataframe(interactive=False)

    outputs = [
        cards_out, brief_out, trend_out, drivers_out,
        cross_out, model_out, data_out, ranking_out, comparison_out
    ]
    run.click(
        render_analysis, inputs=[upload, target, department], outputs=outputs
    )
    app.load(
        render_analysis, inputs=[upload, target, department], outputs=outputs
    )

app.launch(share=True)


/tmp/ipykernel_4624/3741797485.py:505: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1cda200946b215634f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
